# OSU Avatar Feature Creation — final notebook only

**Scope:** This notebook is focused only on **avatar feature creation**. It does not run participant models, does not merge survey data, and does not do robot-session matching.

**Raw input:** image archive(s) placed in `raw_avatar_archives/`.

**Important correction:** this notebook does **not** read any pre-created feature CSV such as old image-metrics inputs, participant-feature files, mapping files, or previously generated outputs. It creates the avatar-level feature table directly from the raw image files and from code shown in this notebook.

Each code block has a clear name. The filename parser is its own standalone section so the extraction of gender, emotion, color, attire, logo, and variant from the image filenames is easy to find and audit.

## Code Block 01 — Setup, paths, and helper functions

This block defines the working paths and reusable helpers. The notebook expects raw image archives in `raw_avatar_archives/` and writes a new feature CSV to `outputs/`.

In [ ]:
from __future__ import annotations

import csv
import math
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

BASE_DIR = Path.cwd()
RAW_ARCHIVE_DIR = BASE_DIR / "raw_avatar_archives"
EXTRACT_DIR = BASE_DIR / "work" / "extracted_avatar_images"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_CSV = OUTPUT_DIR / "OSU_avatar_features_created_by_notebook.csv"

OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_EXTENSIONS = {".jpeg", ".jpg", ".png"}

BAD_PROXY_COLUMNS_THAT_SHOULD_NOT_EXIST = {
    "eyelashes_present_visual_proxy",
    "visible_iris_present_visual_proxy",
    "skin_shading_present_visual_proxy",
    "blush_present_visual_proxy",
    "eyebrows_present_visual_proxy",
    "visible_teeth_present_visual_proxy",
    "mechanical_neck_present_visual_proxy",
    "panel_seams_present_visual_proxy",
    "headset_present_template_code",
    "name_badge_present_visual_proxy",
    "tie_bow_tie_present_filename_rule",
    "tie_bow_tie_proxy",
    "perceived_age_proxy",
}

def write_csv(path: Path, rows: list[dict], headers: list[str] | None = None) -> None:
    if headers is None:
        headers = list(rows[0].keys()) if rows else []
    with path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

def safe_float(value, default=0.0) -> float:
    try:
        number = float(value)
        if math.isfinite(number):
            return number
    except Exception:
        pass
    return default

def percentile_ranks(values: list[float]) -> np.ndarray:
    """Return 0-100 percentile ranks without depending on scipy."""
    array = np.asarray(values, dtype=float)
    n = len(array)
    if n == 0:
        return np.array([])
    if n == 1:
        return np.array([100.0])
    order = np.argsort(array, kind="mergesort")
    ranks = np.empty(n, dtype=float)
    ranks[order] = np.linspace(0, 100, n)
    return ranks

def three_level(value: float, low_cut: float, high_cut: float,
                labels=("low", "medium", "high")) -> str:
    value = safe_float(value)
    if value < low_cut:
        return labels[0]
    if value < high_cut:
        return labels[1]
    return labels[2]

print(f"Working directory: {BASE_DIR}")
print(f"Raw archive directory: {RAW_ARCHIVE_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")

## Code Block 02 — Find and extract raw avatar images

This block extracts raw avatar images from ZIP archives. It does not use an already-created metrics file. It also filters out macOS resource-fork files such as `__MACOSX/._...`, because those are not real images.

In [ ]:
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

archive_paths = sorted(RAW_ARCHIVE_DIR.glob("*.zip"))
if not archive_paths:
    raise FileNotFoundError(
        f"No .zip files found in {RAW_ARCHIVE_DIR}. Put the raw avatar image archive(s) there and rerun."
    )

image_paths: list[Path] = []
archive_image_counts: dict[str, int] = {}

for archive_path in archive_paths:
    target_dir = EXTRACT_DIR / archive_path.stem
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(target_dir)

    real_images = []
    for path in target_dir.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if "__MACOSX" in path.parts or path.name.startswith("._"):
            continue
        real_images.append(path)

    real_images = sorted(real_images)
    image_paths.extend(real_images)
    archive_image_counts[archive_path.name] = len(real_images)
    print(f"{archive_path.name}: extracted {len(real_images)} real images")

print(f"Total real images found: {len(image_paths)}")
if not image_paths:
    raise RuntimeError("No real image files were found after extraction.")

## Code Block 03 — Parse image filenames into avatar attributes

This is the explicit filename-parsing code. It extracts attributes directly from the image filename pattern:

`avatar_{type}_{gender}_{emotion}_{color}_{attire}_{logo}_{variant}.jpeg`

Example: `avatar_humanoid_female_happy_black_formal_nologo_v1.jpeg`.

In [ ]:
AVATAR_FILENAME_PATTERN = re.compile(
    r"^avatar_(?P<type>[^_]+)_(?P<gender>[^_]+)_(?P<emotion>[^_]+)_"
    r"(?P<color>[^_]+)_(?P<attire>[^_]+)_(?P<logo>[^_]+)_"
    r"(?P<variant>v(?P<variant_number>\d+))\.(?P<extension>jpe?g|png)$",
    re.IGNORECASE,
)

def parse_avatar_filename(image_path: Path) -> dict | None:
    """Parse one avatar filename into structured attributes.

    Returns None if the file does not match the expected pattern. The notebook flags
    those rows instead of guessing.
    """
    match = AVATAR_FILENAME_PATTERN.match(image_path.name)
    if match is None:
        return None

    parts = {key: value.lower() for key, value in match.groupdict().items()}
    logo_code = parts["logo"]
    avatar_name = image_path.stem.removeprefix("avatar_")

    return {
        "avatar_name": avatar_name,
        "image_file_name": image_path.name,
        "source_archive_or_folder": image_path.parent.name,
        "avatar_type_filename": parts["type"],
        "avatar_gender_filename": parts["gender"],
        "avatar_emotion_filename": parts["emotion"],
        "avatar_color_filename": parts["color"],
        "avatar_attire_filename": parts["attire"],
        "avatar_logo_code_filename": logo_code,
        "avatar_variant_filename": parts["variant"],
        "avatar_variant_number": int(parts["variant_number"]),
        "logo_present_filename": int(logo_code != "nologo"),
        "formal_attire_filename": int(parts["attire"] == "formal"),
        "happy_emotion_filename": int(parts["emotion"] == "happy"),
    }

parsed_rows = []
unparsed_files = []
seen_names = set()
duplicate_names = []

for image_path in sorted(image_paths):
    parsed = parse_avatar_filename(image_path)
    if parsed is None:
        unparsed_files.append(str(image_path.relative_to(BASE_DIR)))
        continue
    if parsed["avatar_name"] in seen_names:
        duplicate_names.append(parsed["avatar_name"])
    seen_names.add(parsed["avatar_name"])
    parsed["_image_path"] = image_path
    parsed_rows.append(parsed)

print(f"Parsed filenames: {len(parsed_rows)}")
print(f"Unparsed filenames: {len(unparsed_files)}")
print(f"Duplicate avatar names: {len(duplicate_names)}")
if unparsed_files:
    print("Unparsed examples:", unparsed_files[:10])
if duplicate_names:
    print("Duplicate examples:", duplicate_names[:10])

if not parsed_rows:
    raise RuntimeError("No filenames matched the avatar naming pattern.")

## Code Block 04 — Compute objective pixel and detector metrics with OpenCV

This block reads each raw image and computes objective image metrics directly from pixels. It does not infer specific visual objects such as eyelashes, blush, name badges, ties, mechanical necks, or panel seams unless those objects are actually detected or manually coded elsewhere.

In [ ]:
face_cascade = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"))
eye_cascade = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / "haarcascade_eye.xml"))
smile_cascade = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / "haarcascade_smile.xml"))

if face_cascade.empty() or eye_cascade.empty() or smile_cascade.empty():
    raise RuntimeError("OpenCV Haar cascade files could not be loaded.")

def shannon_entropy(gray_image: np.ndarray) -> float:
    histogram = cv2.calcHist([gray_image], [0], None, [256], [0, 256]).ravel()
    probabilities = histogram / histogram.sum()
    probabilities = probabilities[probabilities > 0]
    return float(-(probabilities * np.log2(probabilities)).sum())

def extract_pixel_metrics(row: dict) -> dict:
    path = row["_image_path"]
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")

    bgr = cv2.resize(bgr, (256, 256), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)

    y_channel, cr, cb = cv2.split(ycrcb)
    natural_skin_mask = (cr > 133) & (cr < 180) & (cb > 77) & (cb < 135) & (y_channel > 50)

    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    gradient = np.sqrt(grad_x ** 2 + grad_y ** 2)

    local_mean = cv2.blur(gray.astype(np.float32), (7, 7))
    local_square_mean = cv2.blur(gray.astype(np.float32) ** 2, (7, 7))
    local_sd = np.sqrt(np.maximum(local_square_mean - local_mean ** 2, 0))

    hue_mask = (hsv[:, :, 1] > 30) & (hsv[:, :, 2] > 30)
    if hue_mask.any():
        hue_angle = hsv[:, :, 0][hue_mask].astype(float) * 2 * np.pi / 180
        weights = hsv[:, :, 1][hue_mask].astype(float) / 255
        circular_mean = np.sum(weights * np.exp(1j * hue_angle)) / np.sum(weights)
        hue_concentration = float(abs(circular_mean))
    else:
        hue_concentration = 0.0

    crop = gray[:, 32:224]
    left = crop[:, :96]
    right = np.fliplr(crop[:, 96:192])
    symmetry_proxy = 1 - float(np.mean(np.abs(left.astype(float) - right.astype(float))) / 255)
    symmetry_proxy = max(0.0, min(1.0, symmetry_proxy))

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(35, 35))
    face_detected = int(len(faces) > 0)
    eye_count = 0
    smile_detected = 0
    if face_detected:
        # Use the largest detected face region for eye/smile detection.
        x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
        face_roi = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(face_roi, scaleFactor=1.1, minNeighbors=5, minSize=(8, 8))
        smiles = smile_cascade.detectMultiScale(face_roi, scaleFactor=1.7, minNeighbors=20, minSize=(15, 10))
        eye_count = min(2, len(eyes))
        smile_detected = int(len(smiles) > 0)

    return {
        "image_width_px": int(bgr.shape[1]),
        "image_height_px": int(bgr.shape[0]),
        "mean_saturation_0_255": round(float(hsv[:, :, 1].mean()), 4),
        "mean_brightness_0_255": round(float(hsv[:, :, 2].mean()), 4),
        "hue_concentration_0_1": round(hue_concentration, 6),
        "natural_skin_tone_pixel_fraction_proxy": round(float(natural_skin_mask.mean()), 6),
        "symmetry_proxy_0_1": round(symmetry_proxy, 6),
        "face_detected_visual": face_detected,
        "eye_count_detected_0_2": int(eye_count),
        "eyes_detected_visual": int(eye_count > 0),
        "smile_detected_visual": smile_detected,
        "laplacian_variance": round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 6),
        "flat_fraction": round(float((gradient < 8).mean()), 6),
        "fine_texture_fraction": round(float((gradient > 35).mean()), 6),
        "entropy_bits": round(shannon_entropy(gray), 6),
        "contrast_sd": round(float(gray.std()), 6),
        "edge_density": round(float((cv2.Canny(gray, 100, 200) > 0).mean()), 6),
        "local_texture": round(float(local_sd.mean()), 6),
    }

avatar_rows = []
for parsed in parsed_rows:
    row = {key: value for key, value in parsed.items() if key != "_image_path"}
    row.update(extract_pixel_metrics(parsed))
    avatar_rows.append(row)

print(f"Computed pixel metrics for {len(avatar_rows)} avatar images.")
print("Example row keys:", list(avatar_rows[0].keys())[:15], "...")

## Code Block 05 — Create derived feature columns from real metrics

This block creates derived variables only from filename metadata and objective pixel metrics. The invalid proxy columns flagged earlier are deliberately not created.

In [ ]:
# Percentile-rank selected continuous image metrics within the current image set.
rank_source_fields = [
    "mean_saturation_0_255",
    "mean_brightness_0_255",
    "hue_concentration_0_1",
    "natural_skin_tone_pixel_fraction_proxy",
    "symmetry_proxy_0_1",
    "laplacian_variance",
    "flat_fraction",
    "fine_texture_fraction",
    "entropy_bits",
    "contrast_sd",
    "edge_density",
    "local_texture",
]

for field in rank_source_fields:
    ranks = percentile_ranks([safe_float(row[field]) for row in avatar_rows])
    for i, row in enumerate(avatar_rows):
        row[f"{field}_percentile"] = round(float(ranks[i]), 4)

warm_colors = {"red", "orange", "yellow"}
cool_colors = {"blue", "green", "purple"}
neutral_colors = {"black", "white"}
high_arousal_colors = {"red", "orange", "yellow"}
low_arousal_colors = {"blue", "black", "white"}

for row in avatar_rows:
    color = row["avatar_color_filename"]

    if color in warm_colors:
        row["color_temperature_filename"] = "warm"
    elif color in cool_colors:
        row["color_temperature_filename"] = "cool"
    elif color in neutral_colors:
        row["color_temperature_filename"] = "neutral"
    else:
        row["color_temperature_filename"] = "unknown"

    if color in high_arousal_colors:
        row["color_arousal_filename"] = "high"
    elif color in low_arousal_colors:
        row["color_arousal_filename"] = "low"
    else:
        row["color_arousal_filename"] = "medium"

    # Objective, pixel-based proxies. The names include "proxy" because they are
    # algorithmic summaries, not human visual ratings.
    realism_proxy = (
        row["local_texture_percentile"]
        + row["entropy_bits_percentile"]
        + row["fine_texture_fraction_percentile"]
        + (100 - row["flat_fraction_percentile"])
        + (100 - row["mean_saturation_0_255_percentile"])
    ) / 5
    row["realism_texture_proxy_0_100"] = round(realism_proxy, 4)

    craftsmanship_proxy = (
        row["laplacian_variance_percentile"]
        + row["edge_density_percentile"]
        + row["contrast_sd_percentile"]
        + row["entropy_bits_percentile"]
        + row["local_texture_percentile"]
    ) / 5
    row["craftsmanship_complexity_proxy_0_100"] = round(craftsmanship_proxy, 4)

    human_likeness_proxy = (
        25 * row["face_detected_visual"]
        + 20 * min(row["eye_count_detected_0_2"], 2) / 2
        + 25 * row["natural_skin_tone_pixel_fraction_proxy_percentile"] / 100
        + 20 * row["symmetry_proxy_0_1_percentile"] / 100
        + 10 * row["smile_detected_visual"]
    )
    row["human_likeness_pixel_proxy_0_100"] = round(human_likeness_proxy, 4)

    # Style rule: entropy separates very flat illustrations from images with continuous
    # 3D-style shading. This is a proxy rule, not a manual style judgment.
    row["dimension_style_entropy_rule_proxy"] = (
        "2D/flat" if row["entropy_bits"] < 5.63 else "3D/depth-rendered"
    )
    row["dimension_3d_entropy_rule_binary"] = int(row["dimension_style_entropy_rule_proxy"] == "3D/depth-rendered")

    row["realism_level_proxy"] = three_level(row["realism_texture_proxy_0_100"], 33.333, 66.667,
                                             labels=("cartoonish/low-texture", "semi-realistic", "realistic/high-texture"))
    row["craftsmanship_level_proxy"] = three_level(row["craftsmanship_complexity_proxy_0_100"], 33.333, 66.667)
    row["human_likeness_level_proxy"] = three_level(row["human_likeness_pixel_proxy_0_100"], 33.333, 66.667)

print("Derived features created from filename metadata and objective pixel metrics.")

## Code Block 06 — Quality-control checks

This block checks that the notebook did not create the invalid proxy columns and that the expected filename/image logic worked cleanly.

In [ ]:
all_columns = list(avatar_rows[0].keys())
for row in avatar_rows[1:]:
    for key in row:
        if key not in all_columns:
            all_columns.append(key)

bad_columns_found = sorted(BAD_PROXY_COLUMNS_THAT_SHOULD_NOT_EXIST.intersection(all_columns))
if bad_columns_found:
    raise AssertionError(f"Invalid proxy columns should not be present: {bad_columns_found}")

if unparsed_files:
    raise AssertionError(f"Some image filenames did not parse: {unparsed_files[:10]}")

if duplicate_names:
    raise AssertionError(f"Duplicate avatar names found: {duplicate_names[:10]}")

required_columns = [
    "avatar_name",
    "avatar_gender_filename",
    "avatar_emotion_filename",
    "avatar_color_filename",
    "avatar_attire_filename",
    "avatar_logo_code_filename",
    "avatar_variant_number",
    "logo_present_filename",
    "formal_attire_filename",
    "face_detected_visual",
    "eyes_detected_visual",
    "smile_detected_visual",
    "realism_texture_proxy_0_100",
    "craftsmanship_complexity_proxy_0_100",
]
missing_required = [column for column in required_columns if column not in all_columns]
if missing_required:
    raise AssertionError(f"Missing required output columns: {missing_required}")

print("QC passed.")
print(f"Rows: {len(avatar_rows)}")
print(f"Columns: {len(all_columns)}")
print(f"Invalid proxy columns present: {len(bad_columns_found)}")
print("Counts by filename gender:", dict(Counter(row["avatar_gender_filename"] for row in avatar_rows)))
print("Counts by filename emotion:", dict(Counter(row["avatar_emotion_filename"] for row in avatar_rows)))
print("Counts by filename color:", dict(Counter(row["avatar_color_filename"] for row in avatar_rows)))
print("2D/3D entropy-rule split:", dict(Counter(row["dimension_style_entropy_rule_proxy"] for row in avatar_rows)))

## Code Block 07 — Save the final avatar-level feature dataset

This is the only output created by this notebook. It is avatar-level only: one row per raw avatar image.

In [ ]:
# Remove internal-only fields if any remain, then write the CSV.
for row in avatar_rows:
    for key in list(row.keys()):
        if key.startswith("_"):
            row.pop(key, None)

# Keep a stable, readable column order: identifiers/filename fields first, then metrics/features.
front_columns = [
    "avatar_name",
    "image_file_name",
    "source_archive_or_folder",
    "avatar_type_filename",
    "avatar_gender_filename",
    "avatar_emotion_filename",
    "avatar_color_filename",
    "avatar_attire_filename",
    "avatar_logo_code_filename",
    "avatar_variant_filename",
    "avatar_variant_number",
    "logo_present_filename",
    "formal_attire_filename",
    "happy_emotion_filename",
]
remaining_columns = [column for column in all_columns if column not in front_columns and not column.startswith("_")]
final_columns = front_columns + remaining_columns

avatar_rows = sorted(avatar_rows, key=lambda row: row["avatar_name"])
write_csv(OUTPUT_CSV, avatar_rows, final_columns)

print(f"Saved: {OUTPUT_CSV.relative_to(BASE_DIR)}")
print(f"Final shape: {len(avatar_rows)} rows x {len(final_columns)} columns")

## Final verification summary

This summary is saved in the notebook output so the reviewer can immediately see whether the run completed cleanly.

In [ ]:
print("FINAL NOTEBOOK VERIFICATION")
print("- Raw archive files processed:", archive_image_counts)
print(f"- Avatar rows created: {len(avatar_rows)}")
print(f"- Output columns created: {len(final_columns)}")
print(f"- Unparsed filenames: {len(unparsed_files)}")
print(f"- Duplicated avatar names: {len(duplicate_names)}")
print("- Invalid fabricated proxy columns found:", sorted(BAD_PROXY_COLUMNS_THAT_SHOULD_NOT_EXIST.intersection(final_columns)))
print(f"- Output file exists: {OUTPUT_CSV.exists()} ({OUTPUT_CSV.name})")